In [1]:
import xarray as xr
import numpy as np
from pathlib import Path
import gsw
from xarray import open_mfdataset, open_dataset
from matplotlib import pyplot as plt



# Utilities 
def find_grid_files(grid_type, years, nest, model, delta_t, folder = 'output'): 
    experiment_path = Path('/gxfs_work/geomar/smomw355/model_data/ocean-only/')
    nest_part = '1_' if nest else '' 

    if len(grid_type) < 2:
        grid_type = f'grid_{grid_type}'

    grid_files = (experiment_path / model / 'nemo' / folder).glob(f'{nest_part}{model}_{delta_t}_*_*_{grid_type}.nc')
    grid_files = list(grid_files)
    
    result_files = []
    for file in grid_files:
        start_year = int(str(file).split('/')[-1].split('_')[- (3 + grid_type.count('_'))][:4])
        
        if (years[0] is None or start_year >= years[0]) and \
           (years[1] is None or start_year < years[1]):
            result_files.append(file)
    
    return sorted(result_files)

def load_dataset_from_grid_type(grid_type, years, nest, model, delta_t, chunks = {"time_counter" : 2,"y": 2, "x": None}, **kwargs):
    files_list = find_grid_files(grid_type, years, nest, model, delta_t, **kwargs)
    grid = open_mfdataset(
        files_list, # type: ignore
        chunks= chunks,
        combine='by_coords'
    )
    return grid

def load_masks(nest, model):

    if str.startswith(model, 'VIKING20X.L46-KFS003'):
        # All the mask for the long runs VIKING20X.L46-KFS003-2nd, ... are placed in the original experiment
        model = 'VIKING20X.L46-KFS003' 
    
    nest_part = '1_' if nest else ''
    
    mask_mesh = open_dataset(
        f'/gxfs_work/geomar/smomw355/model_data/ocean-only/{model}/nemo/suppl/{nest_part}mesh_mask.nc',
        decode_cf=False,
        chunks={'y': 100, 'x':-1, 'z': -1},
    )

    if model == 'VIKING20X.L46-KFS003' and not nest:
        nest_part = 'ori_'

    mask_glo = open_dataset(
        f'/gxfs_work/geomar/smomw355/model_data/ocean-only/{model}/nemo/suppl/{nest_part}new_maskglo.nc',
        decode_cf=False,
        chunks={'Y':100, 'X':-1, 'z': -1}
    )

    mask_mesh = mask_mesh.squeeze()
    mask_glo = mask_glo.rename({'X':'x', 'Y':'y'}).squeeze() # Rename coordinates to have the same name as the data file
    
    return mask_mesh, mask_glo



In [2]:

import numpy as np
import xarray as xr
from pathlib import Path


cycle_number = 6
smoothing_days = 90
time_smooth = f'{smoothing_days}D'

dask_url = 'tcp://10.0.4.100:8786'

if cycle_number == 1:
    suffix = '1st_7024' 
    model = 'VIKING20X.L46-KFS003'
    model_years_selector = (1970, 2024)
elif cycle_number == 2:
    suffix = '2nd_5824'
    model = 'VIKING20X.L46-KFS003-2nd'
    model_years_selector = (1958, 2024)
elif cycle_number == 3:
    suffix = '3rd_5824'
    model = 'VIKING20X.L46-KFS003-3rd'
    model_years_selector = (1958, 2024)
elif cycle_number == 4:
    suffix = '4th_5824'
    model = 'VIKING20X.L46-KFS003-4th'
    model_years_selector = (1958, 2024)
elif cycle_number == 5:
    suffix = '5th_5824'
    model = 'VIKING20X.L46-KFS003-5th'
    model_years_selector = (1958, 2024)
elif cycle_number == 6:
    suffix = '6th_5824'
    model = 'VIKING20X.L46-KFS003-6th'
    model_years_selector = (1958, 2024)
else:
    assert False, 'Cycle number not recognized'


dataset_path = Path(f'../rapid-geostrophic-reconstruction/datasets/smoothing_{smoothing_days}_days/argo_after_2012/paperdraft/')


In [3]:
from dask.distributed import Client
from dask_jobqueue import SLURMCluster
import os


client = Client(dask_url)
client

<Client: 'tcp://10.0.4.100:8786' processes=398 threads=398, memory=4.34 TiB>

In [4]:
u_grid = load_dataset_from_grid_type('U', model_years_selector, True,model, '1d', chunks={"time_counter": 1, 'y': 100})
u_grid = u_grid.rename({'depthu': 'z'})

In [5]:
mask_mesh, mask_glo = load_masks(True, model)

In [6]:
mask = (mask_mesh.nav_lat > 25) & (mask_mesh.nav_lat < 27) & (mask_mesh.nav_lon > -80) & (mask_mesh.nav_lon < -10)
mask = mask & mask_mesh.tmaskutil
# mask.plot()

In [7]:
x_min = np.abs(mask_mesh.nav_lon.mean('y') - -80).argmin().compute().values[()]
x_max = np.abs(mask_mesh.nav_lon.mean('y') - -10).argmin().compute().values[()]

y_min = np.abs(mask_mesh.nav_lat.mean('x') - 25).argmin().compute().values[()]
y_max = np.abs(mask_mesh.nav_lat.mean('x') - 27).argmin().compute().values[()]

u_grid_sliced = u_grid.sozotaux.where(mask_mesh.tmaskutil).isel(x = slice(x_min, x_max), y = slice(y_min, y_max))

In [8]:
u_grid_sliced

<xarray.DataArray 'sozotaux' (time_counter: 24106, y: 45, x: 1370)>
dask.array<getitem, shape=(24106, 45, 1370), dtype=float32, chunksize=(1, 45, 1370), chunktype=numpy.ndarray>
Coordinates:
    nav_lat        (y, x) float32 dask.array<chunksize=(45, 1370), meta=np.ndarray>
    nav_lon        (y, x) float32 dask.array<chunksize=(45, 1370), meta=np.ndarray>
    time_centered  (time_counter) datetime64[ns] dask.array<chunksize=(1,), meta=np.ndarray>
  * time_counter   (time_counter) datetime64[ns] 1958-01-01T12:00:00 ... 2023...
Dimensions without coordinates: y, x
Attributes:
    standard_name:       surface_downward_x_stress
    long_name:           Wind Stress along i-axis
    units:               N/m2
    online_operation:    average
    interval_operation:  240 s
    interval_write:      1 d
    cell_methods:        time: mean (interval: 240 s) time_counter: mean

In [9]:
u_grid_sliced_computed = u_grid_sliced.compute()

In [10]:
u_grid_sliced_computed

<xarray.DataArray 'sozotaux' (time_counter: 24106, y: 45, x: 1370)>
array([[[-0.05190941, -0.0507684 , -0.04922258, ...,         nan,
                 nan,         nan],
        [-0.05208404, -0.05091962, -0.04935486, ...,         nan,
                 nan,         nan],
        [-0.05222536, -0.0510305 , -0.04945343, ...,         nan,
                 nan,         nan],
        ...,
        [-0.04318784, -0.04204535, -0.04071371, ...,         nan,
                 nan,         nan],
        [-0.0418089 , -0.04073168, -0.03953059, ...,         nan,
                 nan,         nan],
        [-0.04038538, -0.03942808, -0.0384549 , ...,         nan,
                 nan,         nan]],

       [[-0.02476564, -0.02349886, -0.02208862, ...,         nan,
                 nan,         nan],
        [-0.02621799, -0.02483224, -0.02330504, ...,         nan,
                 nan,         nan],
        [-0.02781457, -0.02632224, -0.02470196, ...,         nan,
                 nan,         nan],
...
        [ 0.09000639,  0.0903154 ,  0.09010888, ...,         nan,
                 nan,         nan],
        [ 0.09117113,  0.09165381,  0.09153506, ...,         nan,
                 nan,         nan],
        [ 0.0924392 ,  0.09299687,  0.09300998, ...,         nan,
                 nan,         nan]],

       [[ 0.02256321,  0.02249274,  0.02244241, ...,         nan,
                 nan,         nan],
        [ 0.02335643,  0.02323055,  0.0231094 , ...,         nan,
                 nan,         nan],
        [ 0.02419104,  0.02400214,  0.02377648, ...,         nan,
                 nan,         nan],
        ...,
        [ 0.05934712,  0.05896876,  0.05873077, ...,         nan,
                 nan,         nan],
        [ 0.05915008,  0.0588678 ,  0.05878065, ...,         nan,
                 nan,         nan],
        [ 0.05888602,  0.05876981,  0.0589046 , ...,         nan,
                 nan,         nan]]], dtype=float32)
Coordinates:
    nav_lat        (y, x) float32 25.0 25.0 25.0 25.0 ... 26.99 26.99 26.99
    nav_lon        (y, x) float32 -79.23 -79.18 -79.13 ... -10.87 -10.82 -10.77
    time_centered  (time_counter) datetime64[ns] 1958-01-01T12:00:00 ... 2023...
  * time_counter   (time_counter) datetime64[ns] 1958-01-01T12:00:00 ... 2023...
Dimensions without coordinates: y, x
Attributes:
    standard_name:       surface_downward_x_stress
    long_name:           Wind Stress along i-axis
    units:               N/m2
    online_operation:    average
    interval_operation:  240 s
    interval_write:      1 d
    cell_methods:        time: mean (interval: 240 s) time_counter: mean

In [11]:
u_grid_sliced_computed.to_netcdf(f'../rapid-geostrophic-reconstruction/datasets/windstress/sozotaux_2527N_8010W_KFS003-{suffix}.nc')